ETL(Extract - Transform - Load)

Load Data

In [2]:
import pandas as pd
import numpy as np
#%pip install openpyxl

In [ ]:
ela = pd.read_csv(r"C:/Users/pbcsc/Downloads/NCF/Semester_1/DB/DB Proj/EDFacts_2022_District_ela.csv", dtype={'NCESLEAID': str}, usecols=['NCESLEAID', 'SchoolYear','AcademicSubject','AgeGrade','Subgroup','ProficientOrAbove_percent'],na_values=['PS','.'])

math = pd.read_csv(r"C:/Users/pbcsc/Downloads/NCF/Semester_1/DB/DB Proj/EDFacts_2022_District_math.csv", dtype={'NCESLEAID': str}, usecols=['NCESLEAID','SchoolYear','AcademicSubject','AgeGrade','Subgroup','ProficientOrAbove_percent'],na_values=['PS','.'])

finance = pd.read_excel(r"C:/Users/pbcsc/Downloads/NCF/Semester_1/DB/DB Proj/sdf22_2a.xlsx", dtype={'LEAID': str}, usecols=['LEAID', 'NAME','V33','TCURINST','TOTALEXP','YEAR','FIPST']) #make LEAID the primary key that is used for merging with other dataframes

## ELA block 

Column Name	Description

| **`NCESLEAID`** | Unique 7-character NCES Local Education Agency identifier with leading zeros preserved.


| **`SchoolYear`** |	Academic school year of the assessment administration (e.g., '2021-2022').


| **`AcademicSubject`** |	Subject area of the standardized assessment ('Mathematics').


| **`AgeGrade`** |	Grade cohort filtered strictly to 'All Grades' (district-wide summary).


| **`Subgroup`** |	Student demographic cohort filtered strictly to 'All Students in LEA'.


| **`ProficientOrAbove_percent`** | Cleaned percentage of tested students meeting or exceeding state proficiency standards; ranges converted to numeric midpoints and inequality prefixes removed.


| **`is_suppressed`** |	Boolean indicator flagging whether the original record was privacy-masked ('PS', '.') under federal reporting rules.

In [19]:
#ela.head()
ela['is_suppressed'] = ela['ProficientOrAbove_percent'].isin(['PS','.','NA',''])

# to get usable numbers for our anlysis later on instead of having ranges that are strings  
def clean_proficiency(val):

    if pd.isna(val):
        return np.nan

    # Convert to string and clean surrounding spaces
    s = str(val).strip().replace('%', '')

    # deals with suppressed or missing values
    if s in ['PS', '.', 'NA', '']:
        return np.nan

    # deals with ranges 
    if '-' in s:
        try:
            low, high = s.split('-')
            return (float(low) + float(high)) / 2.0
        except ValueError:
            return np.nan

    # for greater-than-or-equal 
    if s.startswith('GE') or s.startswith('>='):
        return float(s.replace('GE', '').replace('>=', ''))

    # Less-than-or-equal (
    if s.startswith('LE') or s.startswith('<='):
        return float(s.replace('LE', '').replace('<=', ''))

    # Standard numeric values
    try:
        return float(s)
    except ValueError:
        return np.nan

ela['ProficientOrAbove_percent'] = ela['ProficientOrAbove_percent'].apply(clean_proficiency)


In [32]:
#gets rid of any empty space we might encounter with the column 'subgroup' & makes one big subgroup because we're not looking at each demographic 
ela['Subgroup'] = ela['Subgroup'].astype(str).str.strip()
ela_clean = ela[ela['Subgroup'] == 'All Students in LEA'].copy()
ela_clean = ela_clean[ela_clean['AgeGrade'] == 'All Grades'].copy()

ela_clean.head()

,SchoolYear,NCESLEAID,AcademicSubject,AgeGrade,Subgroup,ProficientOrAbove_percent,is_suppressed
0,2021-2022,100197,ela,All Grades,All Students in LEA,22.0,False
34,2021-2022,100190,ela,All Grades,All Students in LEA,57.0,False
150,2021-2022,100005,ela,All Grades,All Students in LEA,35.0,False
274,2021-2022,100030,ela,All Grades,All Students in LEA,39.0,False
374,2021-2022,100060,ela,All Grades,All Students in LEA,51.0,False


## Math Block

Column Name	Description


| **`NCESLEAID`** | Unique 7-character NCES Local Education Agency identifier with leading zeros preserved.


| **`SchoolYear`** |	Academic school year of the assessment administration (e.g., '2021-2022').


| **`AcademicSubject`** |	Subject area of the standardized assessment ('Mathematics').


| **`AgeGrade`** |	Grade cohort filtered strictly to 'All Grades' (district-wide summary).


| **`Subgroup`** |	Student demographic cohort filtered strictly to 'All Students in LEA'.


| **`ProficientOrAbove_percent`** | Cleaned percentage of tested students meeting or exceeding state proficiency standards; ranges converted to numeric midpoints and inequality prefixes removed.


| **`is_suppressed`** |	Boolean indicator flagging whether the original record was privacy-masked ('PS', '.') under federal reporting rules.

In [31]:
#cleaning df of unusable rows
math['is_suppressed'] = math['ProficientOrAbove_percent'].isin(['PS','.','NA',''])
math['ProficientOrAbove_percent'] = math['ProficientOrAbove_percent'].apply(clean_proficiency)

#gets rid of any empty space we might encounter with the column 'subgroup' & makes one big subgroup because we're not looking at each demographic 
math['Subgroup'] = math['Subgroup'].astype(str).str.strip()
math_clean = math[math['Subgroup'] == 'All Students in LEA'].copy()

# Filter for All Grades
math_clean = math_clean[math_clean['AgeGrade'] == 'All Grades'].copy()


In [38]:
math_clean.head()

,SchoolYear,NCESLEAID,AcademicSubject,AgeGrade,Subgroup,ProficientOrAbove_percent,is_suppressed
0,2021-2022,100197,math,All Grades,All Students in LEA,5.0,False
34,2021-2022,100190,math,All Grades,All Students in LEA,35.0,False
150,2021-2022,100005,math,All Grades,All Students in LEA,17.0,False
274,2021-2022,100030,math,All Grades,All Students in LEA,24.0,False
374,2021-2022,100060,math,All Grades,All Students in LEA,43.0,False


## Finance Block

| Column Name | Description |

| **`LEAID`** | Unique 7-character NCES Local Education Agency (District) identifier with leading zeros preserved. 


| **`NAME`** | Official legal name of the school district. 


| **`V33`** | Fall student membership count (total student enrollment). 


| **`TCURINST`** | Total current expenditures allocated directly to classroom instruction (teacher salaries, supplies, textbooks). 


| **`TOTALEXP`** | Total operational expenditures across all district functions (instruction, support, administration, operations). 


| **`YEAR`** | Reporting fiscal year (e.g., 22 representing FY 2021–2022). 


| **`FIPST`** | Two-digit Federal Information Processing Standard (FIPS) state code (e.g., 01 for Alabama). 

In [37]:
finance.head()  


,LEAID,FIPST,NAME,YEAR,V33,TOTALEXP,TCURINST
0,0100002,1,Alabama Youth Services,22,-2,-1,-1
1,0100005,1,Albertville City,22,5956,65734000,34668000
2,0100006,1,Marshall County,22,5898,87804000,38199000
3,0100007,1,Hoover City,22,13606,208943000,115929000
4,0100008,1,Madison City,22,12302,173520000,77723000


In [44]:
# to get rid of the negative numbers which have a meaning but will mess with calculations
numeric_cols = ['V33', 'TOTALEXP', 'TCURINST']
for col in numeric_cols:
    finance[col] = pd.to_numeric(finance[col], errors='coerce')
    finance.loc[finance[col] < 0, col] = None


In [50]:
finance_ids = set(finance['LEAID'].dropna())
math_ids = set(math_clean['NCESLEAID'].dropna())
ela_ids = set(ela_clean['NCESLEAID'].dropna())

overlap = finance_ids.intersection(math_ids).intersection(ela_ids)
print(f"Finance Districts: {len(finance_ids):,}")
print(f"Math Districts: {len(math_ids):,}")
print(f"ELA Districts: {len(ela_ids):,}")
print(f"\nMatching Districts(all 3): {len(overlap):,}")

Finance Districts: 19,572
Math Districts: 9,228
ELA Districts: 10,094

Matching Districts(all 3): 7,723


In [ ]:
# importing clean data to a new csv

# Clean Finance CSV
finance_clean = finance[finance['LEAID'].isin(overlap)].copy()
finance_clean.to_csv("clean_district_finances.csv", index=False)

# Clean Math CSV
math_export = math_clean[math_clean['LEAID'].isin(overlap)].copy()
math_export.to_csv("clean_math_assessments.csv", index=False)

#Clean ela CSV
ela_export = ela_clean[ela_clean['LEAID'].isin(overlap)].copy()
ela_export.to_csv("clean_ela_assessments.csv", index=False)


